# Evaluation: Systembezug und Gegenstandsextraktion

Dieses Notebook kombiniert **IAA** und **LLM-Evaluation** für den Systembezug.

## Teil A: Inter-Annotator Agreement (IAA)
Zwei Annotatoren bewerten den Systembezug und die extrahierten Gegenstände.
Cohen's Kappa auf den Systembezug (3 Labels), plus qualitativer Abgleich der Phrasen.

## Teil B: LLM-Evaluation gegen Goldstandard
LLM-Systembezug gegen agreed Goldstandard. Precision/Recall/F1 für LIT-NLIT.
Qualitative Stichprobe der extrahierten Gegenstände.

**Input:**
- XLSX mit **zwei Sheets** (eines pro Annotator), die die LLM-Ausgabe evaluiert haben
- ODER: eine XLSX mit LLM-Annotationen + eine XLSX mit Goldstandard

**Modi:**
- `MODE = 'IAA'` → Zwei Annotatoren-Sheets in einer Datei
- `MODE = 'EVAL'` → LLM-Datei vs. Goldstandard-Datei
- `MODE = 'BOTH'` → Erst IAA, dann Eval gegen agreed Goldstandard

## 1. Setup

In [23]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from collections import Counter
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [24]:
# ===== KONFIGURATION =====

MODE = 'BOTH'  # 'IAA', 'EVAL', oder 'BOTH'

# Für IAA / BOTH: XLSX mit zwei Annotator-Sheets
IAA_FILE = "Beutin_complete_INF+System_EVAL_V1.xlsx"  # <-- Hier anpassen
SHEET_A1 = 0  # Erstes Sheet (Index oder Name)
SHEET_A2 = 1  # Zweites Sheet

# Für EVAL: Separate Dateien
LLM_FILE = "Beutin_complete_INF+System_classify_systembezug.xlsx"  # <-- LLM-Output
GOLD_FILE = "Beutin_systembezug_gold.xlsx"  # <-- Goldstandard

# Spaltennamen
COL_PID = "Propositions-ID"
COL_SYS = "Systembezug"
COL_LIT = "LIT-Gegenstand"
COL_NLIT = "NLIT-Gegenstand"

# Ausgabe
OUTPUT_FILE = "systembezug_eval_ergebnisse.xlsx"

EMPTY = "_LEER_"

In [25]:
def interpret_kappa(k):
    if k < 0:     return "schlecht (< 0)"
    if k < 0.21:  return "gering (0.00–0.20)"
    if k < 0.41:  return "ausreichend (0.21–0.40)"
    if k < 0.61:  return "moderat (0.41–0.60)"
    if k < 0.81:  return "substanziell (0.61–0.80)"
    return "fast perfekt (0.81–1.00)"


def load_and_align(df1, df2, label1='A1', label2='A2'):
    """Lädt zwei DataFrames, aligniert über Propositions-ID, normalisiert."""
    # Spalten finden
    for target in [COL_PID, COL_SYS, COL_LIT, COL_NLIT]:
        for label, df in [(label1, df1), (label2, df2)]:
            if target not in df.columns:
                matches = [c for c in df.columns if target.lower() in c.lower()]
                if matches:
                    df.rename(columns={matches[0]: target}, inplace=True)
    
    # Filter und Index
    df1 = df1[df1[COL_PID].notna() & (df1[COL_PID] != '')].set_index(COL_PID)
    df2 = df2[df2[COL_PID].notna() & (df2[COL_PID] != '')].set_index(COL_PID)
    common = df1.index.intersection(df2.index)
    
    print(f"  {label1}: {len(df1)}, {label2}: {len(df2)}, Gemeinsam: {len(common)}")
    
    df1 = df1.loc[common].copy()
    df2 = df2.loc[common].copy()
    
    # Normalisierung Systembezug
    for df in [df1, df2]:
        df[COL_SYS] = df[COL_SYS].fillna(EMPTY).astype(str).str.strip().str.upper()
        df[COL_SYS] = df[COL_SYS].replace({'': EMPTY, 'NAN': EMPTY, 'NONE': EMPTY})
    
    return df1, df2

## 2. Teil A: Inter-Annotator Agreement

In [26]:
if MODE in ('IAA', 'BOTH'):
    df_a1 = pd.read_excel(IAA_FILE, sheet_name=SHEET_A1)
    df_a2 = pd.read_excel(IAA_FILE, sheet_name=SHEET_A2)
    
    xls = pd.ExcelFile(IAA_FILE)
    name1 = xls.sheet_names[SHEET_A1] if isinstance(SHEET_A1, int) else SHEET_A1
    name2 = xls.sheet_names[SHEET_A2] if isinstance(SHEET_A2, int) else SHEET_A2
    
    print(f"IAA: {name1} vs. {name2}")
    df_a1, df_a2 = load_and_align(df_a1, df_a2, name1, name2)
    
    y1 = df_a1[COL_SYS].values
    y2 = df_a2[COL_SYS].values
    
    kappa_iaa = cohen_kappa_score(y1, y2)
    agree_iaa = np.mean(y1 == y2)
    
    labels = sorted(set(y1) | set(y2))
    cm_iaa = confusion_matrix(y1, y2, labels=labels)
    cm_iaa_df = pd.DataFrame(cm_iaa,
        index=[f"{name1}:{l}" for l in labels],
        columns=[f"{name2}:{l}" for l in labels])
    
    print(f"\n{'='*60}")
    print(f"INTER-ANNOTATOR AGREEMENT: SYSTEMBEZUG")
    print(f"{'='*60}")
    print(f"Cohen's Kappa:       {kappa_iaa:.3f} — {interpret_kappa(kappa_iaa)}")
    print(f"%-Übereinstimmung:   {agree_iaa*100:.1f}%")
    print(f"N:                   {len(y1)}")
    print(f"\nKonfusionsmatrix:")
    print(cm_iaa_df.to_string())
    
    # Disagreements
    disagree_mask = y1 != y2
    n_disagree = disagree_mask.sum()
    print(f"\nDisagreements: {n_disagree}")
    if n_disagree > 0:
        for idx in df_a1.index[disagree_mask]:
            prop = df_a1.loc[idx, 'Proposition'] if 'Proposition' in df_a1.columns else ''
            print(f"  {idx}: {name1}={df_a1.loc[idx, COL_SYS]}, {name2}={df_a2.loc[idx, COL_SYS]}")
            if prop:
                print(f"    {str(prop)[:80]}")
else:
    print("IAA übersprungen (MODE != IAA/BOTH)")

IAA: Annotator_1 vs. Annotator_2
  Annotator_1: 106, Annotator_2: 106, Gemeinsam: 106

INTER-ANNOTATOR AGREEMENT: SYSTEMBEZUG
Cohen's Kappa:       1.000 — fast perfekt (0.81–1.00)
%-Übereinstimmung:   100.0%
N:                   106

Konfusionsmatrix:
                      Annotator_2:LIT  Annotator_2:LIT-NLIT  Annotator_2:NLIT
Annotator_1:LIT                     3                     0                 0
Annotator_1:LIT-NLIT                0                    10                 0
Annotator_1:NLIT                    0                     0                93

Disagreements: 0


In [27]:
# IAA: Gegenstandsextraktion (qualitativer Abgleich bei LIT-NLIT)
if MODE in ('IAA', 'BOTH'):
    litnlit_mask = (df_a1[COL_SYS] == 'LIT-NLIT') & (df_a2[COL_SYS] == 'LIT-NLIT')
    n_agreed_litnlit = litnlit_mask.sum()
    
    print(f"\n{'='*60}")
    print(f"GEGENSTANDSEXTRAKTION: QUALITATIVER ABGLEICH")
    print(f"{'='*60}")
    print(f"Beide sagen LIT-NLIT: {n_agreed_litnlit}")
    
    if n_agreed_litnlit > 0 and COL_LIT in df_a1.columns and COL_LIT in df_a2.columns:
        print(f"\n{'PID':<12} {name1+' LIT':<25} {name2+' LIT':<25} {'Match':>6}")
        print(f"{'':>12} {name1+' NLIT':<25} {name2+' NLIT':<25}")
        print('-' * 75)
        for idx in df_a1.index[litnlit_mask]:
            l1 = str(df_a1.loc[idx, COL_LIT] if pd.notna(df_a1.loc[idx, COL_LIT]) else '')[:24]
            l2 = str(df_a2.loc[idx, COL_LIT] if pd.notna(df_a2.loc[idx, COL_LIT]) else '')[:24]
            n1 = str(df_a1.loc[idx, COL_NLIT] if pd.notna(df_a1.loc[idx, COL_NLIT]) else '')[:24]
            n2 = str(df_a2.loc[idx, COL_NLIT] if pd.notna(df_a2.loc[idx, COL_NLIT]) else '')[:24]
            match = '✓' if l1.lower() == l2.lower() and n1.lower() == n2.lower() else '~' if l1[:10].lower() == l2[:10].lower() else '✗'
            print(f"{idx:<12} {l1:<25} {l2:<25} {match:>6}")
            print(f"{'':>12} {n1:<25} {n2:<25}")
    elif n_agreed_litnlit > 0:
        print("  (LIT-Gegenstand/NLIT-Gegenstand Spalten nicht in beiden Sheets vorhanden)")


GEGENSTANDSEXTRAKTION: QUALITATIVER ABGLEICH
Beide sagen LIT-NLIT: 10
  (LIT-Gegenstand/NLIT-Gegenstand Spalten nicht in beiden Sheets vorhanden)


In [28]:
# Goldstandard aus IAA konstruieren
if MODE == 'BOTH':
    agree_mask = df_a1[COL_SYS] == df_a2[COL_SYS]
    df_gold = df_a1[agree_mask].copy()
    df_gold['gold_sys'] = df_gold[COL_SYS]
    
    print(f"\nGoldstandard aus IAA: {len(df_gold)} agreed annotations")
    print(f"  LIT: {(df_gold['gold_sys'] == 'LIT').sum()}")
    print(f"  NLIT: {(df_gold['gold_sys'] == 'NLIT').sum()}")
    print(f"  LIT-NLIT: {(df_gold['gold_sys'] == 'LIT-NLIT').sum()}")
    print(f"  Ausgeschlossen (Disagreement): {(~agree_mask).sum()}")


Goldstandard aus IAA: 106 agreed annotations
  LIT: 3
  NLIT: 93
  LIT-NLIT: 10
  Ausgeschlossen (Disagreement): 0


## 3. Teil B: LLM-Evaluation gegen Goldstandard

In [29]:
if MODE == 'EVAL':
    df_llm_raw = pd.read_excel(LLM_FILE)
    df_gold_raw = pd.read_excel(GOLD_FILE)
    print(f"EVAL: LLM vs. Goldstandard")
    df_llm_eval, df_gold_eval = load_and_align(df_llm_raw, df_gold_raw, 'LLM', 'Gold')
    y_llm = df_llm_eval[COL_SYS].values
    y_gold = df_gold_eval[COL_SYS].values
    gold_pids = df_gold_eval.index

elif MODE == 'BOTH':
    # LLM-Werte aus Annotator-1-Sheet (enthält die LLM-Annotationen)
    # Goldstandard = agreed annotations aus IAA
    gold_pids = df_gold.index
    y_gold = df_gold['gold_sys'].values
    # LLM-Systembezug aus der LLM-Datei laden
    if Path(LLM_FILE).exists():
        df_llm_raw = pd.read_excel(LLM_FILE)
        if COL_PID not in df_llm_raw.columns:
            matches = [c for c in df_llm_raw.columns if COL_PID.lower() in c.lower()]
            if matches:
                df_llm_raw.rename(columns={matches[0]: COL_PID}, inplace=True)
        df_llm_raw = df_llm_raw[df_llm_raw[COL_PID].notna()].set_index(COL_PID)
        common_eval = gold_pids.intersection(df_llm_raw.index)
        y_llm = df_llm_raw.loc[common_eval, COL_SYS].fillna(EMPTY).astype(str).str.strip().str.upper().values
        y_gold = df_gold.loc[common_eval, 'gold_sys'].values
        gold_pids = common_eval
        print(f"\nEVAL: LLM vs. agreed Goldstandard ({len(gold_pids)} Propositionen)")
    else:
        print(f"WARNUNG: {LLM_FILE} nicht gefunden. Verwende Annotator-1-Werte als LLM-Proxy.")
        y_llm = df_a1.loc[gold_pids, COL_SYS].values

else:  # IAA only
    print("Evaluation übersprungen (MODE = IAA)")
    y_llm, y_gold, gold_pids = None, None, None


EVAL: LLM vs. agreed Goldstandard (106 Propositionen)


In [30]:
if y_llm is not None:
    # Accuracy und Kappa
    accuracy = np.mean(y_llm == y_gold)
    kappa_eval = cohen_kappa_score(y_gold, y_llm)
    
    labels_eval = sorted(set(y_llm) | set(y_gold))
    cm_eval = confusion_matrix(y_gold, y_llm, labels=labels_eval)
    cm_eval_df = pd.DataFrame(cm_eval,
        index=[f"Gold:{l}" for l in labels_eval],
        columns=[f"LLM:{l}" for l in labels_eval])
    
    print(f"\n{'='*60}")
    print(f"LLM-PERFORMANCE: SYSTEMBEZUG")
    print(f"{'='*60}")
    print(f"Accuracy:            {accuracy*100:.1f}%")
    print(f"Cohen's Kappa:       {kappa_eval:.3f} — {interpret_kappa(kappa_eval)}")
    print(f"N:                   {len(y_llm)}")
    print(f"\nKonfusionsmatrix (Gold × LLM):")
    print(cm_eval_df.to_string())
    
    # Pro-Label P/R/F1
    print(f"\n{'Label':<12} {'Gold':>5} {'LLM':>5} {'TP':>5} {'Prec':>7} {'Rec':>7} {'F1':>7}")
    print('-' * 55)
    label_rows = []
    for lab in labels_eval:
        m_gold = y_gold == lab
        m_llm = y_llm == lab
        tp = (m_gold & m_llm).sum()
        fp = (~m_gold & m_llm).sum()
        fn = (m_gold & ~m_llm).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        display = '(leer)' if lab == EMPTY else lab
        print(f"{display:<12} {m_gold.sum():>5} {m_llm.sum():>5} {tp:>5} {prec:>7.3f} {rec:>7.3f} {f1:>7.3f}")
        label_rows.append({'Label': display, 'Gold': m_gold.sum(), 'LLM': m_llm.sum(),
                           'TP': tp, 'Precision': round(prec, 3),
                           'Recall': round(rec, 3), 'F1': round(f1, 3)})


LLM-PERFORMANCE: SYSTEMBEZUG
Accuracy:            96.2%
Cohen's Kappa:       0.823 — fast perfekt (0.81–1.00)
N:                   106

Konfusionsmatrix (Gold × LLM):
               LLM:LIT  LLM:LIT-NLIT  LLM:NLIT
Gold:LIT             2             1         0
Gold:LIT-NLIT        0             8         2
Gold:NLIT            1             0        92

Label         Gold   LLM    TP    Prec     Rec      F1
-------------------------------------------------------
LIT              3     3     2   0.667   0.667   0.667
LIT-NLIT        10     9     8   0.889   0.800   0.842
NLIT            93    94    92   0.979   0.989   0.984


In [31]:
# LIT-NLIT Detailanalyse
if y_llm is not None:
    gold_ln = y_gold == 'LIT-NLIT'
    llm_ln = y_llm == 'LIT-NLIT'
    
    tp = (gold_ln & llm_ln).sum()
    fp = (~gold_ln & llm_ln).sum()
    fn = (gold_ln & ~llm_ln).sum()
    
    prec_ln = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec_ln = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_ln = 2 * prec_ln * rec_ln / (prec_ln + rec_ln) if (prec_ln + rec_ln) > 0 else 0
    
    print(f"\n{'='*60}")
    print(f"LIT-NLIT ERKENNUNG (Verknüpfungsstellen)")
    print(f"{'='*60}")
    print(f"Precision: {prec_ln:.3f}")
    print(f"Recall:    {rec_ln:.3f}")
    print(f"F1-Score:  {f1_ln:.3f}")
    print(f"TP={tp}, FP={fp}, FN={fn}")
    
    if fn > 0:
        print(f"\nÜbersehene Verknüpfungsstellen (FN):")
        for i, pid in enumerate(gold_pids[gold_ln & ~llm_ln]):
            print(f"  {pid}: Gold=LIT-NLIT, LLM={y_llm[list(gold_pids).index(pid)]}")
    
    if fp > 0:
        print(f"\nFälschliche Verknüpfungsstellen (FP):")
        for i, pid in enumerate(gold_pids[~gold_ln & llm_ln]):
            print(f"  {pid}: Gold={y_gold[list(gold_pids).index(pid)]}, LLM=LIT-NLIT")


LIT-NLIT ERKENNUNG (Verknüpfungsstellen)
Precision: 0.889
Recall:    0.800
F1-Score:  0.842
TP=8, FP=1, FN=2

Übersehene Verknüpfungsstellen (FN):
  P(005.1): Gold=LIT-NLIT, LLM=NLIT
  P(020.7): Gold=LIT-NLIT, LLM=NLIT

Fälschliche Verknüpfungsstellen (FP):
  P(032.4): Gold=LIT, LLM=LIT-NLIT


## 4. Fehleranalyse

In [32]:
if y_llm is not None:
    error_mask = y_llm != y_gold
    error_rows = []
    
    print(f"\n{'='*60}")
    print(f"FEHLERANALYSE: {error_mask.sum()} Fehler")
    print(f"{'='*60}")
    
    # Verwechslungspaare
    pairs = Counter()
    for i in range(len(y_llm)):
        if y_llm[i] != y_gold[i]:
            pairs[(y_gold[i], y_llm[i])] += 1
    print(f"\nVerwechslungspaare (Gold → LLM):")
    for (g, l), c in pairs.most_common():
        print(f"  {g} → {l}: {c}×")
    
    # Einzelfehler
    print(f"\nAlle Fehler:")
    for i, pid in enumerate(gold_pids):
        if y_llm[i] != y_gold[i]:
            print(f"  {pid}: Gold={y_gold[i]}, LLM={y_llm[i]}")
            error_rows.append({
                'Propositions-ID': pid,
                'Gold': y_gold[i], 'LLM': y_llm[i],
            })
    df_error_export = pd.DataFrame(error_rows)


FEHLERANALYSE: 4 Fehler

Verwechslungspaare (Gold → LLM):
  LIT-NLIT → NLIT: 2×
  NLIT → LIT: 1×
  LIT → LIT-NLIT: 1×

Alle Fehler:
  P(001.1): Gold=NLIT, LLM=LIT
  P(005.1): Gold=LIT-NLIT, LLM=NLIT
  P(020.7): Gold=LIT-NLIT, LLM=NLIT
  P(032.4): Gold=LIT, LLM=LIT-NLIT


## 5. Export

In [33]:
from openpyxl.styles import Font, PatternFill, Alignment

summary_rows = []
if MODE in ('IAA', 'BOTH'):
    summary_rows.extend([
        {'Metrik': 'Inter-Annotator Agreement', 'Wert': '', 'Detail': ''},
        {'Metrik': "  Cohen's Kappa", 'Wert': f'{kappa_iaa:.3f}', 'Detail': interpret_kappa(kappa_iaa)},
        {'Metrik': '  %-Übereinstimmung', 'Wert': f'{agree_iaa*100:.1f}%', 'Detail': f'N={len(y1)}'},
        {'Metrik': '  Disagreements', 'Wert': str(n_disagree), 'Detail': ''},
        {'Metrik': '', 'Wert': '', 'Detail': ''},
    ])

if y_llm is not None:
    summary_rows.extend([
        {'Metrik': 'LLM-Performance', 'Wert': '', 'Detail': ''},
        {'Metrik': '  Accuracy', 'Wert': f'{accuracy*100:.1f}%', 'Detail': f'N={len(y_llm)}'},
        {'Metrik': "  Cohen's Kappa", 'Wert': f'{kappa_eval:.3f}', 'Detail': interpret_kappa(kappa_eval)},
        {'Metrik': '', 'Wert': '', 'Detail': ''},
        {'Metrik': 'LIT-NLIT Erkennung', 'Wert': '', 'Detail': ''},
        {'Metrik': '  Precision', 'Wert': f'{prec_ln:.3f}', 'Detail': ''},
        {'Metrik': '  Recall', 'Wert': f'{rec_ln:.3f}', 'Detail': ''},
        {'Metrik': '  F1-Score', 'Wert': f'{f1_ln:.3f}', 'Detail': f'TP={tp}, FP={fp}, FN={fn}'},
    ])

df_summary = pd.DataFrame(summary_rows)

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    df_summary.to_excel(writer, index=False, sheet_name='Zusammenfassung')
    
    if MODE in ('IAA', 'BOTH'):
        cm_iaa_df.to_excel(writer, sheet_name='CM IAA')
    
    if y_llm is not None:
        cm_eval_df.to_excel(writer, sheet_name='CM LLM vs Gold')
        pd.DataFrame(label_rows).to_excel(writer, index=False, sheet_name='Pro-Label-Metriken')
        if len(df_error_export) > 0:
            df_error_export.to_excel(writer, index=False, sheet_name='Fehler')
    
    hf = PatternFill(start_color='2B4C7E', end_color='2B4C7E', fill_type='solid')
    for ws in writer.sheets.values():
        for cell in ws[1]:
            cell.fill = hf
            cell.font = Font(name='Arial', size=10, bold=True, color='FFFFFF')
            cell.alignment = Alignment(wrap_text=True, vertical='top')
        ws.auto_filter.ref = ws.dimensions

print(f"Exportiert: {OUTPUT_FILE}")

Exportiert: systembezug_eval_ergebnisse.xlsx
